In [2]:
from azure.ai.formrecognizer import DocumentAnalysisClient
from azure.core.credentials import AzureKeyCredential
from azure.storage.blob import BlobServiceClient
from dotenv import load_dotenv
import os
load_dotenv()


# Azure configs
endpoint = os.getenv('ENDPOINT')
key = os.getenv('KEY')
connection_string = os.getenv('CONN_STRING')
container_name = "shivdocstorage"

# Clients
doc_client = DocumentAnalysisClient(endpoint, AzureKeyCredential(key))
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

container_client = blob_service_client.get_container_client(container_name)

results = []

for blob in container_client.list_blobs():
    blob_client = container_client.get_blob_client(blob.name)
    data = blob_client.download_blob().readall()

    poller = doc_client.begin_analyze_document(
        "prebuilt-document",
        document=data
    )
    result = poller.result()

    extracted = {
        "file_name": blob.name,
        "student_name": None,
        "student_id": None,
        "course": None,
        "date": None,
        "fee_amount": None
    }

    # Extract key-value pairs
    if result.key_value_pairs:
        for kv in result.key_value_pairs:
            key = kv.key.content.lower() if kv.key else ""
            value = kv.value.content if kv.value else ""

            if "name" in key:
                extracted["student_name"] = value
            elif "id" in key:
                extracted["student_id"] = value
            elif "course" in key:
                extracted["course"] = value
            elif "date" in key:
                extracted["date"] = value
            elif "fee" in key or "amount" in key:
                extracted["fee_amount"] = value

    results.append(extracted)

# Save JSON
import json
with open("output.json", "w") as f:
    json.dump(results, f, indent=4)

print("Extraction complete!")

Extraction complete!
